# JC THE HOLY OG — ULTIMATE MAX · COLAB NVIDIA GPU

This notebook creates a **separate maximum-graphics copy of the current GitHub game**. It does not modify the public game or overwrite an older notebook.

**Start:** Runtime → Change runtime type → GPU (T4 or stronger), then Runtime → Run all. Setup downloads browser dependencies and the verified city assets. Allow roughly 5–12 minutes on a fresh runtime; asset downloads can take longer.

### Two distinct ways to play
- **Colab GPU remote viewer:** Chrome runs on the Colab VM. The notebook verifies that WebGL2 reports NVIDIA before allowing play, then sends JPEG frames and keyboard/mouse input through the notebook's private Colab proxy. The Gradio panel launches and diagnoses this mode.
- **Direct browser MAX mode:** open the optional direct link for normal browser audio and smoother local input. This mode uses **your device's GPU**, not Colab's GPU.

The remote viewer is an experimental screenshot stream, not a low-latency WebRTC cloud-gaming service. 4K increases GPU, CPU, and network costs; streamed frame rate is measured rather than promised. Remote audio, microphone forwarding, and controller forwarding are not included. Keyboard, mouse, clicks, and on-screen controls are included.

This raises the existing game's available graphical settings. It does not invent missing textures, add path tracing, or certify the game as AAA. The real city data, abilities and NPC UI remain the original game.

In [ ]:
#@title 1 · Maximum graphics configuration
RESOLUTION = "Ultimate 1440p" #@param ["Ultimate 1080p", "Ultimate 1440p", "Ultimate 4K"]
STREAM_RADIUS = 2 #@param [1, 2] {type:"raw"}
SHADOW_MAP = 8192 #@param [4096, 8192] {type:"raw"}
SHADOW_CASTERS = 300 #@param {type:"integer"}
JPEG_QUALITY = 88 #@param {type:"slider", min:65, max:95, step:1}
GRADIO_SHARE = True #@param {type:"boolean"}

import os, sys, pathlib, subprocess, json, time, secrets, shutil
try:
    from google.colab import output
except ImportError:
    raise RuntimeError('Open this notebook in Google Colab with a GPU runtime.')
DIMENSIONS={"Ultimate 1080p":(1920,1080),"Ultimate 1440p":(2560,1440),"Ultimate 4K":(3840,2160)}
WIDTH,HEIGHT=DIMENSIONS[RESOLUTION]
assert STREAM_RADIUS in (1,2) and SHADOW_MAP in (4096,8192)
assert 1 <= SHADOW_CASTERS <= 1000 and 65 <= JPEG_QUALITY <= 95
BASE=pathlib.Path('/content/jc-ultimate-max');BASE.mkdir(exist_ok=True)
RENDER_PORT=7861;GRADIO_PORT=7860
SESSION_TOKEN=secrets.token_urlsafe(32)
GRADIO_PASSWORD=secrets.token_urlsafe(16)
print(f'{RESOLUTION} · {(STREAM_RADIUS*2+1)} × {(STREAM_RADIUS*2+1)} city tiles · {SHADOW_MAP} shadow map')
print('Shadow-map size is capped to the GPU maximum. City coverage follows the available real tiles.')


In [ ]:
#@title 2 · Install browser tools and verify the NVIDIA runtime
try:
    gpu=subprocess.check_output(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],text=True).strip()
except Exception as error:
    raise RuntimeError('No NVIDIA GPU. Choose Runtime > Change runtime type > GPU, reconnect, and rerun.') from error
print('Assigned GPU:',gpu)
subprocess.run([sys.executable,'-m','pip','install','-q','gradio>=5,<7','fastapi>=0.115','uvicorn>=0.30','httpx>=0.27','requests>=2.31','playwright>=1.49,<2'],check=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','--no-install-recommends','vulkan-tools','libvulkan1','libegl1','libgbm1'],check=True)
# Install only NVIDIA user-space graphics libraries when missing; do not replace the kernel driver.
import glob
major=gpu.split(',')[1].strip().split('.')[0]
if not glob.glob('/usr/lib/x86_64-linux-gnu/libGLX_nvidia.so*'):
    selected=None
    for package in (f'libnvidia-gl-{major}-server',f'libnvidia-gl-{major}'):
        result=subprocess.run(['apt-cache','policy',package],capture_output=True,text=True)
        if 'Candidate:' in result.stdout and 'Candidate: (none)' not in result.stdout:
            selected=package;break
    if selected:
        print('Installing matching NVIDIA graphics branch:',selected)
        subprocess.run(['apt-get','install','-y','--no-install-recommends',selected],check=True)
    else:
        print('No matching graphics-library package found. The hardware WebGL check will explain whether this runtime can render JC.')
subprocess.run([sys.executable,'-m','playwright','install','--with-deps','chromium'],check=True)
vulkan=subprocess.run(['vulkaninfo','--summary'],capture_output=True,text=True,timeout=45)
print((vulkan.stdout+vulkan.stderr)[-6000:])
print('CUDA/nvidia-smi is not the final graphics test. Cell 5 checks Chrome’s actual WebGL2 renderer.')


In [ ]:
#@title 3 · Fetch the complete JC source, hydrate city assets, and prepare MAX graphics
if not shutil.which('node'):
    subprocess.run(['apt-get','install','-y','nodejs'],check=True)
node_major=int(subprocess.check_output(['node','--version'],text=True).strip().lstrip('v').split('.')[0])
if node_major<18:raise RuntimeError('Node 18+ is required. Upgrade the Colab Node runtime before continuing.')
REPO_DIR=BASE/f'repo-{time.time_ns()}'
subprocess.run(['git','clone','--depth','1','--single-branch','--branch','main','https://github.com/thebigostreets-byte/jc-the-holy-og.git',str(REPO_DIR)],check=True)
required=['scripts/hydrate-assets.mjs','scripts/build-site.mjs','map.html','map-engine.js','cinematic-look.js']
missing=[p for p in required if not (REPO_DIR/p).is_file()]
if missing:raise RuntimeError('Incomplete source checkout: '+', '.join(missing))
print('Source revision:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip())
# Built-in Node scripts run from the REAL repository, never from an empty jc-node folder.
subprocess.run(['node','scripts/hydrate-assets.mjs'],cwd=REPO_DIR,check=True)
subprocess.run(['node','scripts/hydrate-assets.mjs','--verify-only'],cwd=REPO_DIR,check=True)
engine=(REPO_DIR/'map-engine.js').read_text()
if 'renderer.setPixelRatio(' not in engine:raise RuntimeError('The renderer changed; review the MAX patch for this revision.')
import re
engine=re.sub(r'renderer\.setPixelRatio\([^;]+;', 'renderer.setPixelRatio(1);',engine,count=1)
engine=engine.replace('antialias:false','antialias:true')
# Force desktop detail even if the direct viewer is opened on a phone.
engine=re.sub(r'const mobileMap=[^;]+;', 'const mobileMap=false;',engine,count=1)
(REPO_DIR/'map-engine.js').write_text(engine)
cinematic=(REPO_DIR/'cinematic-look.js').read_text()
marker='const a=new THREE.WebGLRenderTarget'
if marker not in cinematic:raise RuntimeError('The cinematic pipeline changed; review the MSAA patch.')
cinematic=cinematic.replace(marker,'rt.samples=4;\n  '+marker,1)
(REPO_DIR/'cinematic-look.js').write_text(cinematic)


In [ ]:
#@title 4 · Write the maximum-graphics bridge and build the game
ULTIMATE_JS="import * as THREE from './three.module.js';\nconst settings=window.JC_ULTIMATE_CONFIG||{shadow:8192,radius:2,casters:300};\nlet revision=-1,configured=false,shadowLight=null;\nfunction configure(){\n const game=window.studio;if(!window.JC_PLAYER_READY||!game)return;\n const {renderer,scene,camera}=game;\n if(!configured){\n  renderer.setPixelRatio(1);renderer.shadowMap.enabled=true;renderer.shadowMap.type=THREE.PCFSoftShadowMap;\n  const lights=[];scene.traverse(o=>{if(o.isDirectionalLight)lights.push(o);});\n  shadowLight=lights.sort((a,b)=>b.intensity-a.intensity)[0];\n  for(const light of lights)light.castShadow=light===shadowLight;\n  if(shadowLight){\n   const size=Math.min(settings.shadow,renderer.capabilities.maxTextureSize);\n   shadowLight.shadow.mapSize.set(size,size);shadowLight.shadow.map?.dispose();shadowLight.shadow.map=null;\n   Object.assign(shadowLight.shadow.camera,{left:-600,right:600,top:600,bottom:-600,near:1,far:4000});\n   shadowLight.shadow.camera.updateProjectionMatrix();shadowLight.shadow.normalBias=.08;\n   scene.add(shadowLight.target);\n  }\n  const range=document.getElementById('streamRange');if(range){range.value=String(settings.radius);range.dispatchEvent(new Event('change'));}\n  configured=true;\n }\n if(shadowLight){shadowLight.position.copy(camera.position).add(new THREE.Vector3(650,900,-420));shadowLight.target.position.copy(camera.position);}\n const next=game.tileRevision?.()||0;\n if(next!==revision){\n  revision=next;const anisotropy=renderer.capabilities.getMaxAnisotropy();\n  scene.traverse(o=>{if(!o.isMesh)return;o.receiveShadow=true;for(const m of Array.isArray(o.material)?o.material:[o.material]){if(!m)continue;for(const key of ['map','normalMap','roughnessMap','bumpMap','envMap'])if(m[key]){m[key].anisotropy=anisotropy;m[key].needsUpdate=true;}}});\n }\n const candidates=[],position=new THREE.Vector3();\n for(const building of game.buildings.values()){\n  building.getWorldPosition(position);candidates.push({building,distance:position.distanceToSquared(camera.position)});\n }\n candidates.sort((a,b)=>a.distance-b.distance);\n for(let i=0;i<candidates.length;i++)for(const mesh of candidates[i].building.children)if(mesh.isMesh)mesh.castShadow=i<settings.casters&&candidates[i].distance<650*650;\n window.JC_ULTIMATE_STATUS={profile:'ULTIMATE MAX',resolution:renderer.domElement.width+' × '+renderer.domElement.height,shadowMap:shadowLight?.shadow.mapSize.x,anisotropy:renderer.capabilities.getMaxAnisotropy(),tiles:game.loaded.size,buildings:game.buildings.size,drawCalls:renderer.info.render.calls,triangles:renderer.info.render.triangles};\n}\nsetInterval(configure,1500);configure();\n"
VIEWER_HTML='<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>JC · Colab GPU</title><style>body{margin:0;background:#090c13;color:#eadba8;font:14px Arial}header{padding:10px;display:flex;gap:12px;align-items:center;flex-wrap:wrap}button{background:#172632;color:#fff0c8;padding:9px;border:1px solid #7f713f;border-radius:6px}canvas{display:block;width:100%;height:auto;max-height:88vh;object-fit:contain;outline:none;touch-action:none}small{color:#bbc7da}.controls{display:flex;gap:5px;padding:8px;flex-wrap:wrap}</style></head><body><header><b>JC · ULTIMATE MAX · COLAB GPU</b><span id="state">Connecting…</span><button id="full">Full screen</button><small>Click the game to focus. Keyboard + mouse are forwarded to Colab. Remote audio/microphone streaming is not included.</small></header><canvas id="view" tabindex="0"></canvas><div class="controls"><button data-key="w">Forward</button><button data-key="s">Back</button><button data-key="a">Left</button><button data-key="d">Right</button><button data-key="f">Flight</button><button data-key="Space">Up</button><button data-key="Shift">Boost</button><button data-key="e">Interact</button><button data-key="q">Ability</button><button data-key="Escape">Cancel</button></div><script>\nconst canvas=document.getElementById(\'view\'),ctx=canvas.getContext(\'2d\'),state=document.getElementById(\'state\');\nconst token=new URLSearchParams(location.search).get(\'token\');\nconst endpoint=name=>{const url=new URL(name,location.href);url.searchParams.set(\'token\',token);return url.href;};\nlet running=true,inflight=0,lastMove=0,frames=0,since=performance.now(),pendingMove=null;\nconst held=new Set();\nasync function input(data){if(inflight>8&&data.type===\'move\')return;inflight++;try{const r=await fetch(endpoint(\'input\'),{method:\'POST\',headers:{\'Content-Type\':\'application/json\'},body:JSON.stringify(data)});if(!r.ok)state.textContent=\'Input connection lost\';}catch{}finally{inflight--;}}\nasync function release(){held.clear();await input({type:\'release\'});}\nasync function frame(){while(running){const started=performance.now();try{const r=await fetch(endpoint(\'frame\')+\'&t=\'+Date.now(),{cache:\'no-store\'});if(!r.ok){state.textContent=await r.text();await new Promise(r=>setTimeout(r,1500));continue;}const image=await createImageBitmap(await r.blob());if(canvas.width!==image.width||canvas.height!==image.height){canvas.width=image.width;canvas.height=image.height;}ctx.drawImage(image,0,0);image.close();frames++;if(performance.now()-since>2000){state.textContent=(frames*1000/(performance.now()-since)).toFixed(1)+\' streamed frames/s\';frames=0;since=performance.now();}}catch(error){state.textContent=\'Reconnecting to the Colab runtime…\';await new Promise(r=>setTimeout(r,1000));}await new Promise(r=>setTimeout(r,Math.max(0,70-(performance.now()-started))));}}\nfunction point(e){const r=canvas.getBoundingClientRect();const scale=Math.min(r.width/canvas.width,r.height/canvas.height),w=canvas.width*scale,h=canvas.height*scale;return{x:Math.max(0,Math.min(canvas.width-1,(e.clientX-r.left-(r.width-w)/2)/scale)),y:Math.max(0,Math.min(canvas.height-1,(e.clientY-r.top-(r.height-h)/2)/scale))};}\ncanvas.onpointerdown=e=>{canvas.focus();canvas.setPointerCapture(e.pointerId);input({type:\'down\',button:e.button,...point(e)});e.preventDefault();};\ncanvas.onpointerup=e=>{input({type:\'up\',button:e.button,...point(e)});e.preventDefault();};canvas.onpointercancel=release;\ncanvas.onpointermove=e=>{if(performance.now()-lastMove<35)return;lastMove=performance.now();input({type:\'move\',...point(e)});};\ncanvas.oncontextmenu=e=>e.preventDefault();canvas.onwheel=e=>{input({type:\'wheel\',dx:e.deltaX,dy:e.deltaY});e.preventDefault();};\ncanvas.onkeydown=e=>{e.preventDefault();if(held.has(e.key))return;held.add(e.key);input({type:\'keyDown\',key:e.key===\' \'?\'Space\':e.key});};\ncanvas.onkeyup=e=>{e.preventDefault();held.delete(e.key);input({type:\'keyUp\',key:e.key===\' \'?\'Space\':e.key});};\nwindow.addEventListener(\'blur\',release);document.addEventListener(\'visibilitychange\',()=>{if(document.hidden)release();});\nfor(const button of document.querySelectorAll(\'[data-key]\')){button.onpointerdown=e=>{button.setPointerCapture(e.pointerId);input({type:\'keyDown\',key:button.dataset.key});};button.onpointerup=()=>input({type:\'keyUp\',key:button.dataset.key});button.onpointercancel=release;}\ndocument.getElementById(\'full\').onclick=()=>document.documentElement.requestFullscreen?.();setInterval(()=>input({type:\'heartbeat\'}),1000);frame();\n</script></body></html>\n'
RENDER_SERVER='import asyncio, json, os, secrets, time, pathlib\nfrom contextlib import asynccontextmanager\nimport httpx\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import HTMLResponse, Response\nfrom fastapi.staticfiles import StaticFiles\nfrom playwright.async_api import async_playwright\n\nROOT=pathlib.Path(os.environ[\'JC_REPO\'])\nTOKEN=os.environ[\'JC_SESSION_TOKEN\']\nPORT=int(os.environ.get(\'JC_RENDER_PORT\',\'7861\'))\nWIDTH=int(os.environ.get(\'JC_WIDTH\',\'2560\')); HEIGHT=int(os.environ.get(\'JC_HEIGHT\',\'1440\'))\nQUALITY=int(os.environ.get(\'JC_JPEG_QUALITY\',\'88\'))\nstate={\'gpu_verified\':False,\'renderer\':None,\'ready\':False,\'error\':None,\'resolution\':[WIDTH,HEIGHT],\'page_errors\':[]}\npage=None;browser=None;pw=None;lock=asyncio.Lock();keys=set();buttons=set();last_input=time.monotonic()\n\ndef authorize(request):\n    supplied=request.query_params.get(\'token\',\'\')\n    if not secrets.compare_digest(supplied,TOKEN):raise HTTPException(403,\'Open the private viewer link from this notebook.\')\n\nasync def release_inputs():\n    if page:\n        for key in list(keys):\n            try:await page.keyboard.up(key)\n            except Exception:pass\n        for button in list(buttons):\n            try:await page.mouse.up(button=button)\n            except Exception:pass\n    keys.clear();buttons.clear()\n\nasync def start_browser():\n    global page,browser,pw,last_input\n    if page and state[\'ready\']:return\n    state.update(ready=False,error=None,gpu_verified=False)\n    if browser:await browser.close()\n    if not pw:pw=await async_playwright().start()\n    errors=[]\n    for backend in (\'vulkan\',\'gl-egl\'):\n        try:\n            args=[\'--no-sandbox\',\'--enable-gpu\',\'--disable-dev-shm-usage\',\'--disable-software-rasterizer\',\'--ignore-gpu-blocklist\',f\'--use-angle={backend}\',\'--use-gl=angle\']\n            if backend==\'vulkan\':args+=[\'--enable-features=Vulkan\',\'--disable-vulkan-surface\']\n            browser=await pw.chromium.launch(headless=True,args=args)\n            context=await browser.new_context(viewport={\'width\':WIDTH,\'height\':HEIGHT},device_scale_factor=1,has_touch=False)\n            page=await context.new_page()\n            await page.goto(\'about:blank\')\n            report=await page.evaluate(\'\'\'() => {const c=document.createElement(\'canvas\'),gl=c.getContext(\'webgl2\');if(!gl)return {available:false};const x=gl.getExtension(\'WEBGL_debug_renderer_info\');const renderer=x?gl.getParameter(x.UNMASKED_RENDERER_WEBGL):gl.getParameter(gl.RENDERER);const maxTextureSize=gl.getParameter(gl.MAX_TEXTURE_SIZE);gl.getExtension(\'WEBGL_lose_context\')?.loseContext();return {available:true,renderer,maxTextureSize};}\'\'\')\n            renderer=report.get(\'renderer\',\'\')\n            if not report.get(\'available\') or \'nvidia\' not in renderer.lower() or any(x in renderer.lower() for x in (\'swiftshader\',\'llvmpipe\',\'software\')):\n                raise RuntimeError(\'Hardware WebGL2 verification failed: \'+repr(report))\n            state.update(gpu_verified=True,renderer=renderer,backend=backend)\n            break\n        except Exception as error:\n            errors.append(str(error))\n            if browser:await browser.close()\n            browser=None;page=None\n    if not page:\n        state[\'error\']=\'Colab has no verified NVIDIA WebGL renderer. Check the Vulkan/driver cell. CUDA availability alone is not sufficient. \'+ \' | \'.join(errors)\n        raise RuntimeError(state[\'error\'])\n    def capture_error(error):\n        state[\'page_errors\'].append(str(error));state[\'page_errors\']=state[\'page_errors\'][-50:]\n    page.on(\'pageerror\',capture_error)\n    await page.goto(f\'http://127.0.0.1:{PORT}/game/map.html?play=1&quality=ultimate&buildingViews=off\',wait_until=\'domcontentloaded\',timeout=90000)\n    try:\n        await page.wait_for_function(\'window.JC_CITY_READY && window.JC_PLAYER_READY\',timeout=180000)\n        await page.wait_for_function(\'window.JC_ULTIMATE_STATUS\',timeout=15000)\n    except Exception:\n        error=await page.evaluate("document.getElementById(\'jcRecovery\')?.innerText || document.getElementById(\'status\')?.innerText || \'Game readiness timed out\'")\n        state[\'error\']=str(error);raise RuntimeError(state[\'error\'])\n    state.update(ready=True,error=None,resolution=[WIDTH,HEIGHT]);last_input=time.monotonic()\n\n@asynccontextmanager\nasync def lifespan(app):\n    yield\n    if browser:await browser.close()\n    if pw:await pw.stop()\n\napp=FastAPI(lifespan=lifespan)\n\n@app.get(\'/status\')\nasync def status(request:Request):\n    authorize(request)\n    report=dict(state)\n    if page and state[\'ready\']:\n        async with lock:\n            report[\'graphics\']=await page.evaluate(\'window.JC_ULTIMATE_STATUS || {}\')\n            report[\'game\']=await page.evaluate(\'({city:!!window.JC_CITY_READY,player:!!window.JC_PLAYER_READY,loading:window.studio?.streamInfo?.().loading,stats:window.studio?.stats?.()})\')\n    return report\n\n@app.post(\'/start\')\nasync def start(request:Request):\n    authorize(request)\n    async with lock:\n        try:await start_browser()\n        except Exception as error:raise HTTPException(503,str(error))\n    return dict(state)\n\n@app.post(\'/restart\')\nasync def restart(request:Request):\n    global WIDTH,HEIGHT,page\n    authorize(request)\n    data=await request.json()\n    resolution=data.get(\'resolution\',[2560,1440])\n    if resolution not in ([1920,1080],[2560,1440],[3840,2160]):raise HTTPException(400,\'Choose 1080p, 1440p or 4K.\')\n    async with lock:\n        await release_inputs();WIDTH,HEIGHT=resolution;state[\'ready\']=False\n        try:await start_browser()\n        except Exception as error:raise HTTPException(503,str(error))\n    return dict(state)\n\n@app.get(\'/viewer\')\nasync def viewer(request:Request):\n    authorize(request)\n    return HTMLResponse(pathlib.Path(__file__).with_name(\'viewer.html\').read_text(),headers={\'Cache-Control\':\'no-store\',\'Referrer-Policy\':\'no-referrer\'})\n\n@app.get(\'/frame\')\nasync def frame(request:Request):\n    authorize(request)\n    if not page or not state[\'ready\']:raise HTTPException(503,state.get(\'error\') or \'Start the Colab GPU renderer first.\')\n    async with lock:\n        if time.monotonic()-last_input>4:await release_inputs()\n        pixels=await page.screenshot(type=\'jpeg\',quality=QUALITY,timeout=15000,animations=\'allow\')\n    return Response(pixels,media_type=\'image/jpeg\',headers={\'Cache-Control\':\'no-store\'})\n\n@app.post(\'/input\')\nasync def input_event(request:Request):\n    global last_input\n    authorize(request)\n    if not page or not state[\'ready\']:raise HTTPException(503,\'The game is not ready.\')\n    raw=await request.body()\n    if len(raw)>2048:raise HTTPException(413,\'Input too large.\')\n    data=json.loads(raw);kind=data.get(\'type\');last_input=time.monotonic()\n    if kind==\'heartbeat\':return {\'ok\':True}\n    async with lock:\n        if kind==\'release\':await release_inputs()\n        elif kind in (\'move\',\'down\',\'up\'):\n            x=max(0,min(WIDTH-1,float(data.get(\'x\',0))));y=max(0,min(HEIGHT-1,float(data.get(\'y\',0))))\n            await page.mouse.move(x,y)\n            if kind in (\'down\',\'up\'):\n                button={0:\'left\',1:\'middle\',2:\'right\'}.get(data.get(\'button\'), \'left\')\n                if kind==\'down\':await page.mouse.down(button=button);buttons.add(button)\n                else:await page.mouse.up(button=button);buttons.discard(button)\n        elif kind==\'wheel\':await page.mouse.wheel(max(-2000,min(2000,float(data.get(\'dx\',0)))),max(-2000,min(2000,float(data.get(\'dy\',0)))))\n        elif kind in (\'keyDown\',\'keyUp\'):\n            key=data.get(\'key\',\'\')\n            allowed={\'ArrowUp\',\'ArrowDown\',\'ArrowLeft\',\'ArrowRight\',\'Space\',\'Shift\',\'Control\',\'Alt\',\'Enter\',\'Escape\',\'Tab\',\'Backspace\',\'Delete\',\'Home\',\'End\',\'PageUp\',\'PageDown\'}\n            if not isinstance(key,str) or not (len(key)==1 or key in allowed):raise HTTPException(400,\'Unsupported key.\')\n            if kind==\'keyDown\':await page.keyboard.down(key);keys.add(key)\n            else:await page.keyboard.up(key);keys.discard(key)\n        else:raise HTTPException(400,\'Unsupported input.\')\n    return {\'ok\':True}\n\n@app.post(\'/api/npc-chat\')\nasync def npc_chat(request:Request):\n    # Preserve JC\'s existing NPC service without putting API keys in the notebook.\n    body=await request.body()\n    if len(body)>12000:raise HTTPException(413,\'NPC message too large.\')\n    async with httpx.AsyncClient(timeout=25) as client:\n        result=await client.post(\'https://jc-the-holy-og.ill5299.chatgpt.site/api/npc-chat\',content=body,headers={\'content-type\':\'application/json\',\'origin\':\'https://jc-the-holy-og.ill5299.chatgpt.site\'})\n    return Response(result.content,status_code=result.status_code,media_type=\'application/json\')\n\napp.mount(\'/game\',StaticFiles(directory=str(ROOT/\'dist/client\'),html=True),name=\'game\')\nif __name__==\'__main__\':\n    import uvicorn\n    uvicorn.run(app,host=\'127.0.0.1\',port=PORT,log_level=\'warning\')\n'

(REPO_DIR/'colab-ultimate.js').write_text(ULTIMATE_JS)
settings={'shadow':SHADOW_MAP,'radius':STREAM_RADIUS,'casters':SHADOW_CASTERS}
html=(REPO_DIR/'map.html').read_text()
script='<script>window.JC_ULTIMATE_CONFIG='+json.dumps(settings)+'</script><script type="module" src="./colab-ultimate.js"></script>'
html=html.replace('</body>',script+'</body>')
(REPO_DIR/'map.html').write_text(html)
assets_path=REPO_DIR/'scripts/public-assets.json'
assets=json.loads(assets_path.read_text())
if 'colab-ultimate.js' not in assets:assets.append('colab-ultimate.js')
assets_path.write_text(json.dumps(assets,indent=2))
subprocess.run(['node','scripts/build-site.mjs'],cwd=REPO_DIR,check=True)
subprocess.run(['node','scripts/validate-site.mjs'],cwd=REPO_DIR,check=True)
for path in ['dist/client/map.html','dist/client/colab-ultimate.js','dist/client/tiles/C15_R14.glb.gz']:
    if not (REPO_DIR/path).is_file():raise RuntimeError('Missing deployable game file: '+path)
(BASE/'viewer.html').write_text(VIEWER_HTML)
(BASE/'render_server.py').write_text(RENDER_SERVER)
print('Complete game built:',REPO_DIR/'dist/client')


In [ ]:
#@title 5 · Start Colab rendering and verify actual NVIDIA WebGL2
import requests
if 'JC_SERVER' in globals() and JC_SERVER.poll() is None:
    JC_SERVER.terminate()
    try:JC_SERVER.wait(timeout=15)
    except subprocess.TimeoutExpired:JC_SERVER.kill()
server_env=os.environ.copy()
server_env.update(JC_REPO=str(REPO_DIR),JC_SESSION_TOKEN=SESSION_TOKEN,JC_RENDER_PORT=str(RENDER_PORT),JC_WIDTH=str(WIDTH),JC_HEIGHT=str(HEIGHT),JC_JPEG_QUALITY=str(JPEG_QUALITY))
LOG_PATH=BASE/'renderer.log'
JC_LOG_HANDLE=open(LOG_PATH,'w')
JC_SERVER=subprocess.Popen([sys.executable,str(BASE/'render_server.py')],env=server_env,stdout=JC_LOG_HANDLE,stderr=subprocess.STDOUT,cwd=BASE)
LOCAL=f'http://127.0.0.1:{RENDER_PORT}'
def bridge(method,path,payload=None,timeout=240):
    r=requests.request(method,LOCAL+'/'+path,params={'token':SESSION_TOKEN},json=payload,timeout=timeout)
    if not r.ok:raise RuntimeError(r.text[:6000])
    return r.json()
for attempt in range(60):
    if JC_SERVER.poll() is not None:raise RuntimeError(LOG_PATH.read_text()[-8000:])
    try:bridge('GET','status',timeout=2);break
    except Exception:time.sleep(.5)
else:raise RuntimeError('Render server did not start. '+LOG_PATH.read_text()[-8000:])
try:
    GPU_REPORT=bridge('POST','start')
    print(json.dumps(GPU_REPORT,indent=2))
except Exception as error:
    GPU_REPORT=bridge('GET','status',timeout=10)
    print('REMOTE GPU MODE NOT READY:',error)
    print('Renderer log:',LOG_PATH.read_text()[-5000:])
    print('The control panel can retry. Direct browser mode remains available and uses your own GPU.')
# Proxy URLs are discovered through Colab's API, never constructed from an old 5173 host.
PROXY_BASE=output.eval_js(f'google.colab.kernel.proxyPort({RENDER_PORT})')
from urllib.parse import urljoin,urlencode
REMOTE_URL=urljoin(PROXY_BASE,'viewer')+'?'+urlencode({'token':SESSION_TOKEN})
DIRECT_URL=urljoin(PROXY_BASE,'game/map.html?play=1&quality=ultimate&buildingViews=off')
print('NVIDIA graphics verified:',GPU_REPORT.get('gpu_verified',False))


In [ ]:
#@title 6 · Launch the Gradio GPU control panel and the private game viewer
import gradio as gr,html as html_module
if 'JC_DEMO' in globals():
    try:JC_DEMO.close()
    except Exception:pass
def diagnostics():
    try:
        report=bridge('GET','status',timeout=30)
        report['nvidia_smi']=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.used,memory.total,utilization.gpu','--format=csv,noheader'],text=True).strip()
        return json.dumps(report,indent=2)
    except Exception as error:return str(error)+'\n'+LOG_PATH.read_text()[-5000:]
def start_remote():
    try:return json.dumps(bridge('POST','start'),indent=2)
    except Exception as error:return str(error)
def change_resolution(choice):
    try:return json.dumps(bridge('POST','restart',{'resolution':list(DIMENSIONS[choice])}),indent=2)
    except Exception as error:return str(error)
def clear_held_keys():
    try:
        r=requests.post(LOCAL+'/input',params={'token':SESSION_TOKEN},json={'type':'release'},timeout=10)
        r.raise_for_status();return 'Movement and mouse buttons released.'
    except Exception as error:return str(error)
with gr.Blocks(title='JC · Ultimate Max · NVIDIA Colab') as JC_DEMO:
    gr.Markdown('# JC THE HOLY OG · ULTIMATE MAX GPU')
    gr.Markdown('Remote mode is enabled only after NVIDIA WebGL2 verification. The private viewer forwards keyboard/mouse to Colab. The direct link uses your device GPU and supports normal browser audio.')
    gr.HTML('<p><a href="'+html_module.escape(REMOTE_URL,quote=True)+'" target="_blank" rel="noopener">OPEN COLAB GPU GAME</a> &nbsp; | &nbsp; <a href="'+html_module.escape(DIRECT_URL,quote=True)+'" target="_blank" rel="noopener">OPEN DIRECT BROWSER MAX</a></p>')
    with gr.Row():
        start_button=gr.Button('Start / retry NVIDIA renderer',variant='primary')
        check_button=gr.Button('GPU + graphics diagnostics')
        release_button=gr.Button('Release stuck movement')
    resolution=gr.Dropdown(list(DIMENSIONS),value=RESOLUTION,label='Remote render resolution')
    apply_button=gr.Button('Apply resolution and restart remote game')
    report=gr.Textbox(label='Actual renderer, GPU usage, city status and errors',lines=18,value=diagnostics())
    start_button.click(start_remote,outputs=report)
    check_button.click(diagnostics,outputs=report)
    release_button.click(clear_held_keys,outputs=report)
    apply_button.click(change_resolution,inputs=resolution,outputs=report)
    gr.Markdown('Higher resolution and more detailed tiles can lower frame rate. Changing resolution restarts the remote session. Remote session saves are temporary unless exported. Colab idle disconnects end the renderer.')
JC_DEMO.queue(default_concurrency_limit=1)
launch_kwargs=dict(server_name='127.0.0.1',server_port=GRADIO_PORT,share=GRADIO_SHARE,prevent_thread_lock=True,show_error=True)
if GRADIO_SHARE:
    launch_kwargs['auth']=('jc',GRADIO_PASSWORD)
    print('Private Gradio login — username: jc | password:',GRADIO_PASSWORD)
    print('The Gradio share link is password-protected. The game viewer additionally requires your Colab session.')
JC_DEMO.launch(**launch_kwargs)
from IPython.display import display,HTML
safe_url=html_module.escape(REMOTE_URL,quote=True)
display(HTML('<p><a target="_blank" href="'+safe_url+'">Open the private Colab GPU viewer in a full tab</a></p><iframe src="'+safe_url+'" width="100%" height="780" allow="fullscreen" style="border:0"></iframe>'))


## Controls and expected limits

Click the game image before using the keyboard. The game's existing key bindings remain authoritative; the touch controls forward keyboard inputs. Mouse clicks interact with the original HUD, NPCs, menus, and game canvas. Use **Release stuck movement** after a interrupted connection.

**GPU evidence:** the diagnostics panel must show `gpu_verified: true` and an NVIDIA renderer string. It also shows real GPU utilization, active tiles, triangles and draw calls. A successful `nvidia-smi` alone does not prove the game renderer uses the GPU.

**If GPU startup fails:** inspect Cell 2's Vulkan output and Cell 5's renderer error. Some Colab machines expose CUDA without compatible browser graphics libraries. Reconnect to another GPU runtime and rerun. This notebook never silently calls SwiftShader or CPU rendering “NVIDIA GPU mode.” The optional direct mode uses your own device GPU.

**If the viewer returns 404:** rerun Cells 5 and 6 to discover a fresh Colab proxy URL. Old Colab links expire with the runtime. Keep the notebook running while playing.

**If 4K is slow:** choose 1440p or 1080p in Gradio and click Apply. Lower `STREAM_RADIUS` to 1 and rerun setup for 3 × 3 tiles. The screenshot stream, city parsing, and input transport all have CPU/network overhead even with a strong GPU.

**Audio and NPC voice:** remote audio, microphone relay and controller support are not implemented. Use the direct browser MAX link for audio/microphone features supported by the original game. Text NPC chat is forwarded to the existing JC dialogue endpoint and still depends on that service's availability and quota.

No Google Drive mount is required. Source and generated assets live on the temporary VM; no API key is baked into the notebook. This notebook does not reset, delete, or replace your desktop game's saved data. The remote Chrome session has its own temporary browser storage.

### Sources
- [Google Chrome: GPU-backed headless Chrome in Colab](https://developer.chrome.com/docs/web-platform/webgpu/colab-headless)
- [Chromium: GPU hardware in headless Chrome](https://chromium.googlesource.com/chromium/src/+/HEAD/docs/gpu/using-gpu-hardware-in-headless-chrome.md)
- [Gradio: sharing apps and authentication](https://www.gradio.app/guides/sharing-your-app)
- [JC source repository](https://github.com/thebigostreets-byte/jc-the-holy-og)

The notebook code has been syntax- and integration-checked when authored. A real Colab NVIDIA GPU run must still pass the included runtime verification; the authoring environment did not provide a Colab GPU.